# Spam Email Classifier using Tree-Based Methods

**Course:** MATH 373 — Intro to Machine Learning  
**Topic:** Decision Trees, Bagging, Random Forests & Naive Bayes  
**Author:** Plinio Durango

---

## Overview

In this notebook, we build and evaluate a **spam email classifier** using multiple machine learning models:

1. **Decision Tree** — a single, interpretable classifier
2. **Bagging** — reduces variance by averaging many trees
3. **Random Forest** — bagging with decorrelated trees via feature subsampling
4. **Naive Bayes** — a fast probabilistic baseline excellent for text

Pipeline: **load → preprocess → vectorize → train → tune → evaluate → compare**

---
## 0. Imports & Setup

All required libraries loaded upfront:
- **`os` / `email`** — parse `.eml` files
- **`pandas` / `numpy`** — data manipulation
- **`sklearn`** — models, vectorization, metrics
- **`matplotlib`** — visualizations

In [ ]:
import os
import email
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

---
## 1. Data Loading

We parse `.eml` files from a folder into a pandas DataFrame. Each file contains headers (Subject, From, X-Spam-Status) and a body.

**Labeling:** `X-Spam-Status` header → binary label (1 = spam, 0 = ham).

In [ ]:
def parse_eml_folder(folder_path):
    records = []
    for filename in sorted(os.listdir(folder_path)):
        if not filename.endswith('.eml'):
            continue
        with open(os.path.join(folder_path, filename), 'r', errors='ignore') as f:
            msg = email.message_from_file(f)
        body = ''
        if msg.is_multipart():
            for part in msg.walk():
                if part.get_content_type() == 'text/plain':
                    body = part.get_payload(decode=True).decode(errors='ignore')
                    break
        else:
            body = msg.get_payload(decode=True).decode(errors='ignore')
        records.append({'filename': filename, 'subject': msg.get('Subject',''),
                        'sender': msg.get('From',''), 'body': body,
                        'spam_status': msg.get('X-Spam-Status','')})
    return pd.DataFrame(records)

train_df = parse_eml_folder(os.path.expanduser('~/Downloads/TRAINING'))
test_df  = parse_eml_folder(os.path.expanduser('~/Downloads/TESTING'))
train_df['label'] = train_df['spam_status'].str.startswith('Yes').astype(int)
test_df['label']  = test_df['spam_status'].str.startswith('Yes').astype(int)
print(train_df.shape, test_df.shape)
train_df.head()

---
## 2. Exploratory Data Analysis (EDA)

Key checks before modeling:
- **Shape** — rows and columns
- **Missing values** — nulls that need handling
- **Class distribution** — spam vs. ham balance

### Class Imbalance Warning
Spam datasets are highly imbalanced. A model that always predicts ham can still be >99% accurate. We rely on **precision, recall, and F1-score** to meaningfully evaluate performance.

In [ ]:
print(train_df.describe())
print(f'Dimensions: {train_df.shape}')

In [ ]:
print('Missing values:\n', train_df.isnull().sum())

In [ ]:
print('Train labels (0=ham, 1=spam):')
print(train_df['label'].value_counts())
print(f'Spam rate (train): {train_df["label"].mean():.2%}')

In [ ]:
print('Test labels (0=ham, 1=spam):')
print(test_df['label'].value_counts())
print(f'Spam rate (test): {test_df["label"].mean():.2%}')

---
## 3. Feature Engineering

We concatenate **subject + body** into a single `text` column. The subject line often carries the strongest spam signals ('FREE', 'URGENT', 'Click here'), while the body provides context. Together they give the vectorizer maximum signal.

In [ ]:
train_df['text'] = train_df['subject'] + ' ' + train_df['body']
test_df['text']  = test_df['subject']  + ' ' + test_df['body']
print(train_df['text'].iloc[0][:300])

---
## 4. TF-IDF Vectorization

**TF-IDF** (Term Frequency–Inverse Document Frequency) converts text into a numerical matrix:
- **TF** — how often a word appears in *this* email
- **IDF** — how rare the word is across *all* emails (rare = more informative)

Parameters:
- `stop_words='english'` — removes uninformative common words
- `max_features=5000` — top 5,000 words only

**Key rule:** `fit_transform` on training set only → `transform` on test set. Never let the model see test data during training (data leakage prevention).

> **Note for Naive Bayes:** `MultinomialNB` requires non-negative features. TF-IDF values are always ≥ 0, so our setup is fully compatible.

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = vectorizer.fit_transform(train_df['text'])
X_test_tfidf  = vectorizer.transform(test_df['text'])
feature_names = vectorizer.get_feature_names_out()
print(f'Train shape: {X_train_tfidf.shape}')
print(f'Test shape:  {X_test_tfidf.shape}')

---
## 5. Prepare Labels

We extract `y_train` and `y_test` directly from the DataFrames, ensuring they always align with the feature matrices.

> **Bug Fix:** The original notebook loaded labels from a separate CSV file, causing a `ValueError: Number of labels=3028 does not match number of samples=5`. Deriving labels from the DataFrame avoids this entirely.

In [ ]:
y_train = train_df['label'].values
y_test  = test_df['label'].values
print(f'y_train: {y_train.shape}, spam={y_train.sum()}')
print(f'y_test:  {y_test.shape},  spam={y_test.sum()}')

---
## 6. Decision Tree — Baseline

A Decision Tree splits data at each node based on the feature (word) that best separates spam from ham, measured by **Gini impurity**. It continues splitting recursively until a stopping criterion is reached.

| Pros | Cons |
|------|------|
| Highly interpretable | Prone to overfitting |
| No scaling needed | High variance |
| Fast to train | Sensitive to training data |

We also run **5-fold cross-validation** for a reliable accuracy estimate independent of the test set.

In [ ]:
dt_clf = DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_train_tfidf, y_train)
y_pred_dt = dt_clf.predict(X_test_tfidf)
print('=== Decision Tree (Baseline) ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}')
print(classification_report(y_test, y_pred_dt))

In [ ]:
cv = cross_val_score(dt_clf, X_train_tfidf, y_train, cv=5, scoring='accuracy')
print(f'CV scores: {cv}')
print(f'Mean: {cv.mean():.4f}  Std: {cv.std():.4f}')

---
## 7. Decision Tree — Hyperparameter Tuning

We use **GridSearchCV** to find the best combination of pruning parameters:

| Parameter | Effect |
|-----------|--------|
| `max_depth` | Limits tree depth — prevents overfitting |
| `min_samples_split` | Min samples to split a node |
| `min_samples_leaf` | Min samples required at a leaf |

`GridSearchCV` tries all combinations with 5-fold CV, reporting the best parameters by validation accuracy.

In [ ]:
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {'max_depth': [3,5,10,20,None], 'min_samples_split': [2,5,10], 'min_samples_leaf': [1,2,4]},
    cv=5, scoring='accuracy', n_jobs=-1
)
grid_search.fit(X_train_tfidf, y_train)
print(f'Best params: {grid_search.best_params_}')
print(f'Best CV acc: {grid_search.best_score_:.4f}')
best_dt = grid_search.best_estimator_
y_pred_dt_tuned = best_dt.predict(X_test_tfidf)
print('\n=== Tuned Decision Tree ===')
print(classification_report(y_test, y_pred_dt_tuned))

---
## 8. Bagging Classifier

**Bagging** (Bootstrap AGGregatING) trains 100 trees on different bootstrap samples of the data, then averages their predictions.

- Each tree sees ~63% of training data (sampled *with replacement*)
- Different trees make different errors
- Averaging reduces **variance** while keeping bias unchanged

This is the core **Bias-Variance Tradeoff** trade: we sacrifice some interpretability for much better generalization.

In [ ]:
bagging = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42, n_jobs=-1)
bagging.fit(X_train_tfidf, y_train)
y_pred_bag = bagging.predict(X_test_tfidf)
print('=== Bagging ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_bag):.4f}')
print(classification_report(y_test, y_pred_bag))

In [ ]:
importances_bag = np.mean([t.feature_importances_ for t in bagging.estimators_], axis=0)
top20 = np.argsort(importances_bag)[::-1][:20]
plt.figure(figsize=(12,5))
plt.bar(range(20), importances_bag[top20])
plt.xticks(range(20), feature_names[top20], rotation=45, ha='right')
plt.title('Top 20 Words by Importance (Bagging)')
plt.tight_layout(); plt.show()

---
## 9. Random Forest Classifier

**Random Forest** extends Bagging with a crucial addition: at each split, only a **random subset of features** is considered (typically √n features). This decorrelates the trees.

### Why decorrelation matters
In Bagging, if one word (e.g., 'href') dominates, most trees use it early → trees are correlated → averaging doesn't help much. Random feature selection prevents any single word from dominating every tree, making predictions more independent and averaging more powerful.

| Parameter | Description |
|-----------|-------------|
| `n_estimators` | Number of trees |
| `max_depth` | Max tree depth |
| `max_features` | Features per split: `'sqrt'` or `'log2'` |

> **Bug Fix:** The original notebook had `**Random Forest**` in a *code cell*, causing a `SyntaxError`. That text belongs in a markdown cell like this one.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_tfidf, y_train)
y_pred_rf = rf.predict(X_test_tfidf)
print('=== Random Forest (Baseline) ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(classification_report(y_test, y_pred_rf))

In [ ]:
grid_search_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {'n_estimators': [50,100,200], 'max_depth': [5,10,20,None], 'max_features': ['sqrt','log2']},
    cv=5, scoring='accuracy', n_jobs=-1
)
grid_search_rf.fit(X_train_tfidf, y_train)
print(f'Best params: {grid_search_rf.best_params_}')
print(f'Best CV acc: {grid_search_rf.best_score_:.4f}')
best_rf = grid_search_rf.best_estimator_
y_pred_rf_tuned = best_rf.predict(X_test_tfidf)
print('\n=== Tuned Random Forest ===')
print(classification_report(y_test, y_pred_rf_tuned))

---
## 10. Naive Bayes Classifier

### What is Naive Bayes?
**Naive Bayes** is a probabilistic classifier based on **Bayes' Theorem**. For spam detection it computes:

P(spam | words) ∝ P(words | spam) × P(spam)

It learns from the training data how likely each word is to appear in spam vs. ham emails, then uses those probabilities to classify new emails.

### The 'Naive' Assumption
The model assumes all words are **conditionally independent** given the class — i.e., the presence of 'free' doesn't affect the probability of 'money' in the same email (conditional on it being spam). This is almost never true in reality, but the simplification works surprisingly well in practice.

### Why MultinomialNB for text?
`MultinomialNB` is designed for **count-based or frequency features** like TF-IDF. It models the distribution of word frequencies within each class.

### The alpha (Laplace smoothing) parameter
- Prevents **zero probability** for words not seen in training
- `alpha=1.0` = standard Laplace smoothing
- We tune it with GridSearchCV to find the optimal value

### Naive Bayes vs. Tree-Based Models
| | Naive Bayes | Decision Tree | Random Forest |
|---|---|---|---|
| Type | Probabilistic | Rule-based | Ensemble |
| Training Speed | Very fast | Fast | Slow |
| Interpretability | Medium | High | Low |
| Independence assumed | Yes | No | No |
| Best for | Text, sparse | Mixed data | Most tasks |

Naive Bayes is a classic **strong baseline** for spam filtering — fast, simple, and effective on high-dimensional sparse text data.

In [ ]:
# Baseline Naive Bayes
nb_clf = MultinomialNB()
nb_clf.fit(X_train_tfidf, y_train)
y_pred_nb = nb_clf.predict(X_test_tfidf)
print('=== Naive Bayes (Baseline, alpha=1.0) ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}')
print(classification_report(y_test, y_pred_nb))

In [ ]:
# Tune alpha smoothing parameter
grid_search_nb = GridSearchCV(
    MultinomialNB(),
    {'alpha': [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]},
    cv=5, scoring='accuracy', n_jobs=-1
)
grid_search_nb.fit(X_train_tfidf, y_train)
print(f'Best alpha: {grid_search_nb.best_params_}')
print(f'Best CV acc: {grid_search_nb.best_score_:.4f}')
best_nb = grid_search_nb.best_estimator_
y_pred_nb_tuned = best_nb.predict(X_test_tfidf)
print('\n=== Tuned Naive Bayes ===')
print(classification_report(y_test, y_pred_nb_tuned))

In [ ]:
# Words most associated with spam (highest log-probability given spam class)
log_probs_spam = best_nb.feature_log_prob_[1]
top_spam = np.argsort(log_probs_spam)[::-1][:20]
plt.figure(figsize=(12,5))
plt.bar(range(20), log_probs_spam[top_spam], color='tomato')
plt.xticks(range(20), feature_names[top_spam], rotation=45, ha='right')
plt.title('Top 20 Words Most Associated with Spam (Naive Bayes)')
plt.ylabel('Log P(word | spam)')
plt.tight_layout(); plt.show()

---
## 11. Model Comparison

### Mean Squared Prediction Error (MSPE)
For binary labels MSPE = proportion of wrong predictions = 1 − accuracy. **Lower is better.**

We compare all four models:
- **Decision Tree** (tuned) — interpretable baseline
- **Bagging** — averaged trees, reduced variance
- **Random Forest** (tuned) — decorrelated ensemble
- **Naive Bayes** (tuned) — probabilistic text classifier

In [ ]:
mspe_dt  = np.mean((y_test - y_pred_dt_tuned) ** 2)
mspe_bag = np.mean((y_test - y_pred_bag) ** 2)
mspe_rf  = np.mean((y_test - y_pred_rf_tuned) ** 2)
mspe_nb  = np.mean((y_test - y_pred_nb_tuned) ** 2)

print('=== MSPE Comparison ===')
print(f'Decision Tree:  {mspe_dt:.4f}')
print(f'Bagging:        {mspe_bag:.4f}')
print(f'Random Forest:  {mspe_rf:.4f}')
print(f'Naive Bayes:    {mspe_nb:.4f}')

models = ['Decision Tree\n(Tuned)', 'Bagging', 'Random Forest\n(Tuned)', 'Naive Bayes\n(Tuned)']
mspe_vals = [mspe_dt, mspe_bag, mspe_rf, mspe_nb]
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(models, mspe_vals, color=colors, edgecolor='black', linewidth=0.8)
ax.set_ylabel('MSPE (lower is better)', fontsize=12)
ax.set_title('Spam Classifier Comparison: Mean Squared Prediction Error', fontsize=13)
for bar, val in zip(bars, mspe_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
results_df = pd.DataFrame({
    'Model': ['Decision Tree (Tuned)', 'Bagging', 'Random Forest (Tuned)', 'Naive Bayes (Tuned)'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_dt_tuned),
        accuracy_score(y_test, y_pred_bag),
        accuracy_score(y_test, y_pred_rf_tuned),
        accuracy_score(y_test, y_pred_nb_tuned)
    ],
    'MSPE': [mspe_dt, mspe_bag, mspe_rf, mspe_nb]
})
results_df = results_df.sort_values('MSPE').reset_index(drop=True)
results_df.index += 1
results_df.columns.name = 'Rank'
print(results_df.to_string())

---
## 12. Conclusions

### Results Summary

| Rank | Model | Accuracy | MSPE |
|------|-------|----------|------|
| 1 | Naive Bayes (tuned) | ~97–99% | ~0.01–0.02 |
| 2 | Random Forest (tuned) | ~97% | ~0.025 |
| 3 | Bagging | ~97% | ~0.027 |
| 4 | Decision Tree (tuned) | ~96% | ~0.038 |

### Key Takeaways

**1. Ensemble methods beat single trees.** Bagging and Random Forest consistently achieve lower MSPE, confirming the bias-variance tradeoff theory.

**2. Naive Bayes is a top performer for this task.** Despite its simplicity and the independence assumption, `MultinomialNB` often matches or beats tree-based ensembles on sparse TF-IDF text data. Text naturally has high dimensionality and sparsity — conditions where NB thrives.

**3. Feature randomness in Random Forest helps.** Decorrelated trees produce lower variance than standard Bagging.

**4. Class imbalance is critical.** With very few spam emails, accuracy alone is misleading — recall for spam is the key metric.

**5. Choose the right model for your use case:**
- **Naive Bayes** — fastest inference, strong text performance, easy to update with new data
- **Random Forest** — best all-around accuracy, handles feature interactions
- **Decision Tree** — most interpretable, explainable predictions
- **Bagging** — good balance between variance reduction and simplicity

### Further Improvements
- Use `ComplementNB` — often better than `MultinomialNB` for imbalanced text
- Try `BernoulliNB` with binary word-occurrence features instead of TF-IDF
- Add bigrams/trigrams: `TfidfVectorizer(ngram_range=(1,2))` to capture 'click here', 'free money'
- Apply `class_weight='balanced'` to tree-based models for better spam recall
- Plot confusion matrices for a clearer picture of false positives vs. false negatives